# Week 2-3 Listing Dataset EDA

This notebook documents exploratory data analysis for the CRMLS listing dataset.

Goals:
- Inspect rows, columns, and data types
- Review property types and Residential filtering logic
- Calculate missing counts and missing percentages
- Flag columns with more than 90% missing values
- Separate market analysis fields from metadata fields
- Review numeric distributions and outliers
- Save reports and a filtered analysis dataset

## 1. Import Packages And Set Paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "data" / "reports" / "week2_3_listing_eda"
FIGURE_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

LISTING_FILE = PROCESSED_DIR / "crmls_listings_combined_residential_202401_202606.csv"
LISTING_FILE

## 2. Load The Listing Dataset

This file was created in Week 1 by concatenating monthly listing CSVs and filtering to `PropertyType == "Residential"`.

In [ ]:
listings = pd.read_csv(LISTING_FILE, low_memory=False)

# The combined Week 1 file is already filtered to Residential.
# Keep a separate name because the later EDA cells use listings_residential.
listings_residential = listings.copy()

print(f"Rows: {listings_residential.shape[0]:,}")
print(f"Columns: {listings_residential.shape[1]:,}")
listings_residential.head()


## 3. Inspect Structure

In [ ]:
listings.columns

In [ ]:
dtypes_summary = listings.dtypes.astype(str).reset_index()
dtypes_summary.columns = ["column", "dtype"]
dtypes_summary

## 4. Review Property Types

Because this processed file is already Residential-filtered, we expect only Residential rows here. If you need Residential vs. other property type share, use raw monthly files or save an unfiltered combined listing dataset.

In [ ]:
property_type_counts = listings["PropertyType"].fillna("Missing").value_counts(dropna=False)
property_type_counts

## 5. Missing Value Analysis

In [ ]:
missing_report = pd.DataFrame({
    "column": listings_residential.columns,
    "missing_count": listings_residential.isna().sum().values,
})

missing_report["missing_pct"] = (
    missing_report["missing_count"] / len(listings_residential) * 100
).round(2)
missing_report["over_90_pct_missing"] = missing_report["missing_pct"] > 90
missing_report = missing_report.sort_values("missing_pct", ascending=False)

missing_report.head(25)

In [ ]:
high_missing_columns = missing_report[missing_report["over_90_pct_missing"]]
high_missing_columns

## 6. Market Analysis Fields Vs Metadata Fields

In [ ]:
core_market_fields = {
    "ListingKey", "ListingId", "ListingContractDate", "ContractStatusChangeDate",
    "ListPrice", "OriginalListPrice", "PropertyType", "PropertySubType",
    "LivingArea", "LotSizeAcres", "BedroomsTotal", "BathroomsTotalInteger",
    "DaysOnMarket", "YearBuilt", "CountyOrParish", "City", "PostalCode",
    "Latitude", "Longitude", "MlsStatus",
}

metadata_fields = {
    "ListingKeyNumeric", "ListAgentEmail", "ListAgentFirstName",
    "ListAgentLastName", "ListAgentFullName", "CoListAgentFirstName",
    "CoListAgentLastName", "BuyerAgentMlsId", "BuyerAgentFirstName",
    "BuyerAgentLastName", "ListOfficeName", "BuyerOfficeName",
    "CoListOfficeName", "BuyerOfficeAOR",
}

field_classification = []
for column in listings_residential.columns:
    base_column = column.replace(".1", "")
    if base_column in metadata_fields:
        category = "metadata"
    elif base_column in core_market_fields:
        category = "market_analysis_core"
    else:
        category = "market_analysis_other"
    field_classification.append({"column": column, "field_category": category})

field_classification = pd.DataFrame(field_classification)
field_classification

## 7. Numeric Distribution Summary

In [ ]:
numeric_fields = [
    "ListPrice", "OriginalListPrice", "LivingArea", "LotSizeAcres",
    "BedroomsTotal", "BathroomsTotalInteger", "DaysOnMarket", "YearBuilt",
]

numeric_summary_rows = []

for field in numeric_fields:
    series = pd.to_numeric(listings_residential[field], errors="coerce")
    percentiles = series.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    numeric_summary_rows.append({
        "field": field,
        "count": int(series.count()),
        "missing_count": int(series.isna().sum()),
        "min": series.min(),
        "max": series.max(),
        "mean": series.mean(),
        "median": series.median(),
        "p01": percentiles.loc[0.01],
        "p05": percentiles.loc[0.05],
        "p25": percentiles.loc[0.25],
        "p50": percentiles.loc[0.5],
        "p75": percentiles.loc[0.75],
        "p95": percentiles.loc[0.95],
        "p99": percentiles.loc[0.99],
    })

numeric_summary = pd.DataFrame(numeric_summary_rows)
numeric_summary

## 8. Histograms And Boxplots

In [ ]:
for field in numeric_fields:
    series = pd.to_numeric(listings_residential[field], errors="coerce").dropna()
    if series.empty:
        continue

    p01 = series.quantile(0.01)
    p99 = series.quantile(0.99)
    zoomed_series = series[(series >= p01) & (series <= p99)]
    outside_zoom_count = len(series) - len(zoomed_series)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0, 0].hist(series, bins=50)
    axes[0, 0].set_title(f"{field} Histogram - Full Range")
    axes[0, 0].set_xlabel(field)
    axes[0, 0].set_ylabel("Count")

    axes[0, 1].boxplot(series, vert=False, showfliers=True)
    axes[0, 1].set_title(f"{field} Boxplot - Full Range")
    axes[0, 1].set_xlabel(field)

    axes[1, 0].hist(zoomed_series, bins=50)
    axes[1, 0].set_title(f"{field} Histogram - 1st to 99th Percentile")
    axes[1, 0].set_xlabel(field)
    axes[1, 0].set_ylabel("Count")

    axes[1, 1].boxplot(zoomed_series, vert=False, showfliers=True)
    axes[1, 1].set_title(f"{field} Boxplot - 1st to 99th Percentile")
    axes[1, 1].set_xlabel(field)

    fig.suptitle(
        f"{field}: full range plus zoomed view. "
        f"Rows outside 1st-99th percentile view: {outside_zoom_count:,}",
        y=1.02,
    )
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{field}_full_and_zoomed_histogram_boxplot.png")
    plt.show()

## 9. Extreme Outlier Review

This uses the IQR rule to flag extreme values for later cleaning review. The notebook does not automatically remove outliers.

In [ ]:
outlier_rows = []

for field in numeric_fields:
    series = pd.to_numeric(listings_residential[field], errors="coerce").dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    low_outliers = int((series < lower_bound).sum())
    high_outliers = int((series > upper_bound).sum())
    
    outlier_rows.append({
        "field": field,
        # iqr is interquartile range, iqr = Q3 - Q1. lower_bound = Q1 - 1.5 * IQR, upper_bound = Q3 + 1.5 * IQR
        "iqr_lower_bound": lower_bound,
        "iqr_upper_bound": upper_bound,
        "low_outlier_count": low_outliers,
        "high_outlier_count": high_outliers,
        "total_outlier_count": low_outliers + high_outliers,
        "outlier_pct": round((low_outliers + high_outliers) / len(series) * 100, 2),
        "min": series.min(),
        "max": series.max(),
    })

outlier_report = pd.DataFrame(outlier_rows)
outlier_report

## 10. Answer Listing EDA Questions

In [ ]:
list_price = pd.to_numeric(listings_residential["ListPrice"], errors="coerce")
original_list_price = pd.to_numeric(listings_residential["OriginalListPrice"], errors="coerce")
days_on_market = pd.to_numeric(listings_residential["DaysOnMarket"], errors="coerce")

listing_date = pd.to_datetime(listings_residential["ListingContractDate"], errors="coerce")
contract_status_date = pd.to_datetime(listings_residential["ContractStatusChangeDate"], errors="coerce")

valid_price_pairs = listings_residential[list_price.notna() & original_list_price.notna()].copy()
valid_price_pairs["ListPriceNumeric"] = pd.to_numeric(valid_price_pairs["ListPrice"], errors="coerce")
valid_price_pairs["OriginalListPriceNumeric"] = pd.to_numeric(valid_price_pairs["OriginalListPrice"], errors="coerce")

price_reduced_pct = (valid_price_pairs["ListPriceNumeric"] < valid_price_pairs["OriginalListPriceNumeric"]).mean() * 100
price_increased_pct = (valid_price_pairs["ListPriceNumeric"] > valid_price_pairs["OriginalListPriceNumeric"]).mean() * 100
price_unchanged_pct = (valid_price_pairs["ListPriceNumeric"] == valid_price_pairs["OriginalListPriceNumeric"]).mean() * 100

summary_answers = pd.DataFrame([
    {"question": "Average list price", "answer": list_price.mean()},
    {"question": "Median list price", "answer": list_price.median()},
    {"question": "Average OriginalListPrice", "answer": original_list_price.mean()},
    {"question": "Median OriginalListPrice", "answer": original_list_price.median()},
    {"question": "Average Days on Market", "answer": days_on_market.mean()},
    {"question": "Median Days on Market", "answer": days_on_market.median()},
    {"question": "Current list price below original list price pct", "answer": price_reduced_pct},
    {"question": "Current list price above original list price pct", "answer": price_increased_pct},
    {"question": "Current list price unchanged pct", "answer": price_unchanged_pct},
    {"question": "ContractStatusChangeDate before ListingContractDate rows", "answer": int((contract_status_date < listing_date).sum())},
])

summary_answers

In [ ]:
county_median_list_prices = (
    listings_residential.assign(ListPriceNumeric=list_price)
    .dropna(subset=["CountyOrParish", "ListPriceNumeric"])
    .groupby("CountyOrParish")["ListPriceNumeric"]
    .median()
    .sort_values(ascending=False)
    .reset_index(name="median_list_price")
)

county_median_list_prices.head(10)

## 11. Save Deliverable Outputs

In [ ]:
filtered_output = PROCESSED_DIR / "crmls_listings_residential_week2_3_filtered_202401_202606.csv"

# Rebuild this table if the county-median cell was not run before this save cell.
if "county_median_list_prices" not in globals():
    list_price_for_county = pd.to_numeric(listings_residential["ListPrice"], errors="coerce")
    county_median_list_prices = (
        listings_residential.assign(ListPriceNumeric=list_price_for_county)
        .dropna(subset=["CountyOrParish", "ListPriceNumeric"])
        .groupby("CountyOrParish")["ListPriceNumeric"]
        .median()
        .sort_values(ascending=False)
        .reset_index(name="median_list_price")
    )

listings_residential.to_csv(filtered_output, index=False)
dtypes_summary.to_csv(REPORT_DIR / "dtypes_summary.csv", index=False)
property_type_counts.reset_index().to_csv(REPORT_DIR / "property_type_counts.csv", index=False)
field_classification.to_csv(REPORT_DIR / "field_classification.csv", index=False)
missing_report.to_csv(REPORT_DIR / "missing_value_report.csv", index=False)
numeric_summary.to_csv(REPORT_DIR / "numeric_distribution_summary.csv", index=False)
outlier_report.to_csv(REPORT_DIR / "numeric_outlier_report.csv", index=False)
summary_answers.to_csv(REPORT_DIR / "eda_question_answers.csv", index=False)
county_median_list_prices.to_csv(REPORT_DIR / "county_median_list_prices.csv", index=False)

print("Saved filtered dataset:", filtered_output)
print("Saved reports to:", REPORT_DIR)
print("Saved figures to:", FIGURE_DIR)

